In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
from api.config.vehicle_profile_config import VehicleProfileConfig
from api.config.output_config import OutputConfig
from api.config.segmentation_config import SegmentationConfig

from moviasai.data.utils import load_raw_data

In [3]:
profile_cfg = VehicleProfileConfig.from_yaml('../config/vehicle_profile_config.yaml')
output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')
seg_cfg = SegmentationConfig.from_yaml('../config/segmentation_config.yaml')

DATA_PATH = '../../datasets/full/telemetria_movias2025_features.csv'
df = load_raw_data(DATA_PATH)

In [4]:
from pathlib import Path
from moviasai.profiling.profile import VehicleProfile
from moviasai.profiling.classification import TypeClassifier, SegmentationClassifier
from api.config.data_quality_config import DataQualityConfig

# Configs
dq_cfg = DataQualityConfig.from_yaml('../config/data_quality_config.yaml')

# Classificadores
cls_dir = Path(output_cfg.models.classification)
type_clf = TypeClassifier.load_model(str(cls_dir / 'stage1' / 'stage1_type_BEST.onnx'))
seg_clf_km = SegmentationClassifier.load_model(str(cls_dir / 'stage2' / 'stage2_km_BEST.onnx'))
seg_clf_h = SegmentationClassifier.load_model(str(cls_dir / 'stage2' / 'stage2_h_BEST.onnx'))

# Profile com override_features=True
profile = VehicleProfile.from_config(
    segmentation_config=seg_cfg,
    profile_config=profile_cfg,
    data_quality_config=dq_cfg,
    override_features=True,
    extract_metadata=True
)
profile

✓ Modelo ONNX carregado: C:\Users\f0pi\git\apimovias\models\classification\stage1\stage1_type_BEST.onnx
  Scaler (JSON): C:\Users\f0pi\git\apimovias\models\classification\stage1\stage1_type_BEST_scaler.json
  Metadata: C:\Users\f0pi\git\apimovias\models\classification\stage1\stage1_type_BEST_metadata.json
  Nome: stage1_type
  Features: 3

✓ Modelo ONNX carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.onnx
  Scaler (JSON): C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST_scaler.json
  Metadata: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST_metadata.json
  Nome: stage2_km
  Features: 5

✓ Modelo ONNX carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST.onnx
  Scaler (JSON): C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST_scaler.json
  Metadata: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST_metadata.json
  Nome: stage2_h
  Feature

VehicleProfile(n_vehicles=0, sample_size=168, p_upper=95)

In [14]:
df_long = profile.run_pipeline(
    df=df,
    type_classifier=type_clf,
    segment_classifier_km=seg_clf_km,
    segment_classifier_h=seg_clf_h,
)
df_long

GERANDO FEATURES
Features totais: 153
Veículos: 20169

🗑️  Séries vazias (km=0 e h=0): 10,005 veículos removidos
Dataset carregado: 1,707,552 registros
Veiculos: 10164
Target: km

Dataset carregado: 1,707,552 registros
Veiculos: 10164
Target: h

FILTRANDO DATASET
Target: km
Min weeks: 12
Max gap: 4


🚀 Iniciando construção de janelas...
   Target: km
   Window size: 7 dias (1 semanas)
   Stride: 7 dias
   Max gap: 4

📍 Etapa 1: Identificando períodos contínuos...
   ✓ 16,368 períodos identificados
   ✓ 9,278 veículos únicos

📍 Etapa 2: Gerando segmentos de janelas (vetorizado)...

🔍 GERAÇÃO DE JANELAS (VETORIZADA)
   🔄 Pré-computando semanas ativas...
      ✓ 184,083 semanas ativas para 10,164 veículos (0.04s)

📅 Período disponível:
   min_date: 2025-04-28
   max_date: 2025-10-05
   total_days: 161

📏 Parâmetros:
   window_size: 7 dias (1 semanas)
   stride: 7 dias
   Critério: TODAS as 1 semanas devem ter atividade

📆 Gerando 23 janelas...
   ✓ 23 janelas válidas (dentro do período)



,veiculo_id,feature,feature_class,valor
0,4,cluster_0_h,h,0.006808
1,11,cluster_0_h,h,0.440860
2,38,cluster_0_h,h,0.000264
3,81,cluster_0_h,h,0.000000
4,105,cluster_0_h,h,0.000000
...,...,...,...,...
1303675,29415,type_2,type,0.999992
1303676,29417,type_2,type,0.999646
1303677,29419,type_2,type,0.998662
1303678,29426,type_2,type,0.999104


In [15]:
# Veículos inválidos (sem NaN/Inf) — incluídos no perfil para predição
import pandas as pd

df_invalid = profile.get_invalid_vehicles_long()
print(f"Veículos inválidos com features válidas: {df_invalid['veiculo_id'].nunique()}")
print(f"Linhas: {len(df_invalid)}")

# Concatenar com df_long (válidos + inválidos)
df_long = pd.concat([df_long, df_invalid], ignore_index=True)
print(f"df_long total: {len(df_long)} ({df_long['veiculo_id'].nunique()} veículos)")

Veículos inválidos com features válidas: 2016
Linhas: 308448
df_long total: 1612128 (10164 veículos)


In [16]:
df_invalid[df_invalid['valor'].isna()].head()

,veiculo_id,feature,feature_class,valor


In [7]:
profile.df_sampled

veiculo_id,data,km_dia_clean,h_dia_clean
i64,date,f64,f64
4,2025-04-26,2.7,6.332431
4,2025-04-27,5.2,6.332431
4,2025-04-28,13.8,6.332431
4,2025-04-29,15.3,6.332431
4,2025-04-30,13.4,6.332431
…,…,…,…
29428,2025-10-06,74.303661,1.812747
29428,2025-10-07,74.303661,1.812747
29428,2025-10-08,73.2,1.792778


In [8]:
profile.df_metadata_h.to_csv('../../tmp/metadata_h.csv', index=False)
profile.df_metadata_km.to_csv('../../tmp/metadata_km.csv', index=False)

profile.df_sampled.write_csv('../../tmp/sample.csv')
df_long.to_csv('../../tmp/long.csv', index=False)

In [9]:
df_long[['feature', 'feature_class']].drop_duplicates()

,feature,feature_class
0,cluster_0_h,h
8148,cluster_0_km,km
16296,cluster_1_h,h
24444,cluster_1_km,km
32592,cluster_2_km,km
...,...,...
1262940,seg_p75_km,km
1271088,seg_taxa_dias_ativos_h,h
1279236,seg_taxa_dias_ativos_km,km
1287384,type_1,1


In [10]:
profile.km_extractors['day'].feature_names

['day_1_mean_km',
 'day_1_std_km',
 'day_1_p25_km',
 'day_1_p75_km',
 'day_1_iqr_km',
 'day_1_prob_active_km',
 'day_1_cv_km',
 'day_2_mean_km',
 'day_2_std_km',
 'day_2_p25_km',
 'day_2_p75_km',
 'day_2_iqr_km',
 'day_2_prob_active_km',
 'day_2_cv_km',
 'day_3_mean_km',
 'day_3_std_km',
 'day_3_p25_km',
 'day_3_p75_km',
 'day_3_iqr_km',
 'day_3_prob_active_km',
 'day_3_cv_km',
 'day_4_mean_km',
 'day_4_std_km',
 'day_4_p25_km',
 'day_4_p75_km',
 'day_4_iqr_km',
 'day_4_prob_active_km',
 'day_4_cv_km',
 'day_5_mean_km',
 'day_5_std_km',
 'day_5_p25_km',
 'day_5_p75_km',
 'day_5_iqr_km',
 'day_5_prob_active_km',
 'day_5_cv_km',
 'day_6_mean_km',
 'day_6_std_km',
 'day_6_p25_km',
 'day_6_p75_km',
 'day_6_iqr_km',
 'day_6_prob_active_km',
 'day_6_cv_km',
 'day_7_mean_km',
 'day_7_std_km',
 'day_7_p25_km',
 'day_7_p75_km',
 'day_7_iqr_km',
 'day_7_prob_active_km',
 'day_7_cv_km']

In [11]:
import polars as pl
(profile.df_sampled.filter(pl.col('veiculo_id') == 41) == 0).sum()


veiculo_id,data,km_dia_clean,h_dia_clean
u32,u32,u32,u32
0,0,0,0


In [12]:
profile.df_no_anomalies[profile.df_no_anomalies['veiculo_id'] == 41]

,veiculo_id,razao_km_h,proporcao_km,corr_km_h,seg_cv_gaps_km,seg_taxa_dias_ativos_km,seg_p75_km,seg_iqr_km,day_1_mean_km,day_1_std_km,...,cycle_ratio_fim_inicio_h,quality,quality_reason,type_1,type_2,cluster_0_km,cluster_1_km,cluster_2_km,cluster_0_h,cluster_1_h


In [13]:
df_long['valor']

0          0.004804
1          0.440078
2          0.000177
3          0.000000
4          0.000000
             ...   
1303675    0.000009
1303676    0.000389
1303677    0.001426
1303678    0.000984
1303679    0.000029
Name: valor, Length: 1303680, dtype: float64

In [14]:
f = profile.df_no_anomalies.isna().sum(axis=1) > 0

In [17]:
df_long['valor'].isna().sum()

np.int64(0)